In [2]:
!pip install folium branca matplotlib mapclassify

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Roberto Ponce López\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Pipeline para determinar los links a los que se conectarán las estaciones de BRT/TL mediante walking links

In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np
import win32com.client as com
import os
from shapely import wkt
import folium
import branca
import matplotlib as plt
import mapclassify
from shapely.ops import nearest_points
from shapely.geometry import Point


c:\Users\Roberto Ponce López\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
PATH = r"C:\Users\Roberto Ponce López\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey (1)\Modelación Urbana - Red Vial Guadalajara"
RED_FOLDER = r"Red Base GDL\RedBase Conectores y Atts"

In [160]:
#Red base GDL (con 2,203 zonas)
red_base = os.path.join(PATH, RED_FOLDER, "RedBase 150826 - finalfilter - copia.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

### Read StopPoints

In [161]:
stop_points = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("No")],
    "NumLines_BRT": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("NumLines_TSys(BRT)")],
    "NumLines_TL": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("NumLines_TSys(TL)")],
    "NodeNo": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("NodeNo")],
    "XCoord": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("XCoord")],
    "YCoord": [i[1] for i in Visum.Net.StopPoints.GetMultiAttValues("YCoord")],
})

stop_points = gpd.GeoDataFrame(
    stop_points, 
    geometry=gpd.points_from_xy(stop_points["XCoord"], stop_points["YCoord"]), 
    crs="EPSG:4326"
)

# reproyectar a metros
stop_points = stop_points.to_crs("EPSG:32613")

# keep only stop points of TL & BRT (those that are not connected to the network)
access_stop_points = stop_points[
    (stop_points["NumLines_BRT"] > 0) |
    (stop_points["NumLines_TL"] > 0)
].copy()

print(f"Total {len(stop_points):,} from Visum")
print(f"Out of which {len(access_stop_points):,} serve BRT or TL and need a walking connection")

Total 16,634 from Visum
Out of which 250 serve BRT or TL and need a walking connection


### Read RedVial Visum (nodes+links)

In [ ]:
# Nodes
nodes = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Nodes.GetMultiAttValues("No")],
    "XCoord": [i[1] for i in Visum.Net.Nodes.GetMultiAttValues("XCoord")],
    "YCoord": [i[1] for i in Visum.Net.Nodes.GetMultiAttValues("YCoord")],
})
nodes = gpd.GeoDataFrame(
    nodes, 
    geometry=gpd.points_from_xy(nodes["XCoord"], nodes["YCoord"]), 
    crs="EPSG:4326"
)

# Links
links = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "FromNode": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "ToNode": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "TSysSet": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TSysSet")],
    "TypeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TypeNo")],
    "NumTouchingLineRoutes": [i[1] for i in Visum.Net.Links.GetMultiAttValues("NumTouchingLineRoutes")],
    "GRUPO_CAP": [i[1] for i in Visum.Net.Links.GetMultiAttValues("GRUPO_CAP")],
    "HIGHWAY": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HIGHWAY")],
    "geometry": [i[1] for i in Visum.Net.Links.GetMultiAttValues("WKTPolyWGS84")],
})
links["geometry"] = links["geometry"].apply(wkt.loads)
links = gpd.GeoDataFrame(links, geometry="geometry", crs="EPSG:4326")

# reproyectar a metros
nodes = nodes.to_crs("EPSG:32613")
links = links.to_crs("EPSG:32613")

# crear subconjunto de links validos para conectar las estaciones
# 1) aquellos que tengan valor en HIGHWAY (los que no tienen son los links de BRT y TL)
links_red_vial = links[links["HIGHWAY"] != ""]

# 2) aquellos que no pertenezcan a vias carreteras (y ciclovias porque tampoco se conectan con la red)
unvalid_highways = [
    "trunk", "trunk_link",
    "motorway", "motorway_link",
    "cycleway"
]
links_red_vial = links_red_vial[~links_red_vial["HIGHWAY"].isin(unvalid_highways)]

### __1) Find Candidate Links that intersect buffer__ around stop points
- __RULE 1:__ Links that intersect buffer around Stop Point
(and find distance to each candidate link)

In [75]:
search_stops = access_stop_points.copy()

BUFFER_SEARCH_RADIUS = 100

# buffer around stops
search_stops["geometry"] = search_stops.geometry.buffer(BUFFER_SEARCH_RADIUS)

# valid links that intersect buffer
candidates = gpd.sjoin(
    search_stops,
    links_red_vial,
    how="left",
    predicate="intersects"
)

# keep geometría original del stop (POINT)
candidates["stop_geometry"] = candidates.index.map(
    access_stop_points.geometry
)

# keep geometría del link candidato (LINESTRING)
candidates["link_geometry"] = candidates["index_right"].map(
    links_red_vial.geometry
)

# find nearest point on link
# donde idealmente deberia estar el walking link (esto representa que el nearest_node tendria que ser un nodo nuevo)
# nearest_point returns 2 points ([0] point on first geometry, [1] point on second geometry)
candidates["nearest_point"] = candidates.apply(
    lambda row: nearest_points(
        row["stop_geometry"],
        row["link_geometry"]
    )[1],
    axis=1
)

# Find distance to each candidate link
candidates["distance_to_link"] = candidates.apply(
    lambda row: row["stop_geometry"].distance(
        row["link_geometry"]
    ),
    axis=1
)

### __2) From links that intersect buffer find if there are any pedestrian/footway__
- __REGLA 2:__ buscar primero aquellos que son footway o pedestrian y esten lo suficientemente cerca

In [113]:
PREFERRED_WALK_HIGHWAYS = ["footway","pedestrian"]
MAX_PREFERRED_WALK_DISTANCE = 50

""" 
Find candidate links that are:
 1. OSM-categorized as pedestrian
 2. Within buffer
 3. To a 50m distance
"""
preferred_walk_candidates = candidates[
    candidates["HIGHWAY"].isin(PREFERRED_WALK_HIGHWAYS)
    & (
        candidates["distance_to_link"]
        <= MAX_PREFERRED_WALK_DISTANCE
    )
].copy()

stops_with_preferred_walk = set(
    preferred_walk_candidates["No_left"].unique()
)
print(f"{len(stops_with_preferred_walk)} stop points have an OSM-walking link close enough")

selected_preferred_walk = (
    preferred_walk_candidates
    .sort_values("distance_to_link")
    .groupby("No_left")
    .first()
    .reset_index()
)

remaining_candidates = candidates[
    ~candidates["No_left"].isin(
        stops_with_preferred_walk
    )
].copy()
print(f"{remaining_candidates["No_left"].nunique()} didn't find an OSM-walking link within the buffer")
print(f"And will proceed to next criterion")

145 stop points have an OSM-walking link close enough
105 didn't find an OSM-walking link within the buffer
And will proceed to next criterion


### __3) For links that didn't find a walking link,__ look for parallel partners (stop points that are in front of each other) and need to find a link in each direction

In [114]:
remaining_stop_points = access_stop_points[
    ~access_stop_points["No"].isin(stops_with_preferred_walk)
].copy()
print(f"Look for pairs from the remaining {len(remaining_stop_points)} that didn't find walking link")

# Find stop points that are near to each other
# casos donde estan casi paralelos pero en diferente linea (como in front to each other)
stop_pairs = gpd.sjoin_nearest(
    remaining_stop_points,
    remaining_stop_points,
    how="left",
    max_distance=50,
    distance_col="pair_distance",
    exclusive=True
)

# Make sure it doesn't combine BRT & TL
stop_pairs = stop_pairs[
    ((stop_pairs["NumLines_BRT_left"] > 0) & (stop_pairs["NumLines_BRT_right"] > 0)) |
    ((stop_pairs["NumLines_TL_left"] > 0) & (stop_pairs["NumLines_TL_right"] > 0))
].copy()

# Reciprocal Nearest 
# pares mutuos
mutual_pairs = stop_pairs[
    stop_pairs.apply(
        lambda row: (
            ((stop_pairs["No_left"] == row["No_right"]) &
             (stop_pairs["No_right"] == row["No_left"])).any()
        ),
        axis=1
    )
].copy()

Look for pairs from the remaining 105 that didn't find walking link


In [115]:
mutual_pairs["stop_geometry"] = mutual_pairs.geometry

mutual_pairs["pair_geometry"] = mutual_pairs["index_right"].map(remaining_stop_points.geometry)

mutual_pairs["midpoint"] = mutual_pairs.apply(
    lambda row: Point(
        (
            row["stop_geometry"].x +
            row["pair_geometry"].x
        ) / 2,
        (
            row["stop_geometry"].y +
            row["pair_geometry"].y
        ) / 2
    ),
    axis=1
)

In [116]:
midpoint_by_stop = mutual_pairs.set_index("No_left")["midpoint"]

remaining_candidates["midpoint"] = remaining_candidates["No_left"].map(
    midpoint_by_stop
)

### Encontrar el "lado" del stop point
(para que busque el nearest link pero de su lado)

In [117]:
def is_outward(row):

    if row["midpoint"] is None or pd.isna(row["midpoint"]):
        return None

    stop = row["stop_geometry"]
    midpoint = row["midpoint"]
    candidate = row["nearest_point"]

    # Dirección desde el centro hacia afuera
    outward_x = stop.x - midpoint.x
    outward_y = stop.y - midpoint.y

    # Dirección desde el stop hacia el link candidato
    candidate_x = candidate.x - stop.x
    candidate_y = candidate.y - stop.y

    dot_product = (
        outward_x * candidate_x +
        outward_y * candidate_y
    )

    return dot_product > 0

In [118]:
remaining_candidates["outward"] = remaining_candidates.apply(
    is_outward,
    axis=1
)

In [127]:
paired_remaining_candidates = remaining_candidates[
    remaining_candidates["midpoint"].notna()
].copy()

unpaired_remaining_candidates = remaining_candidates[
    remaining_candidates["midpoint"].isna()
].copy()

# De los paired candidates (stops que estan en frente la una de la otra)
# Quedarnos con los candidate links que estan en la direccion "outwards" del SP (para que no elija links del otro lado aunque esten mas cerca)
paired_with_outward_ids = set(
    paired_remaining_candidates.loc[
        paired_remaining_candidates["outward"] == True,
        "No_left"
    ].unique()
)

paired_outward = paired_remaining_candidates[
    (paired_remaining_candidates["No_left"].isin(paired_with_outward_ids)) &
    (paired_remaining_candidates["outward"] == True)
].copy()
paired_no_outward = paired_remaining_candidates[
    ~paired_remaining_candidates["No_left"].isin(
        paired_with_outward_ids
    )
].copy()


paired_outward["selection_method"] = "outward_nearest"
paired_no_outward["selection_method"] = "no_outward_nearest_fallback"
unpaired_remaining_candidates["selection_method"] = "unpaired_nearest_fallback"

valid_remaining_candidates = pd.concat([
    paired_outward,
    paired_no_outward,
    unpaired_remaining_candidates
], ignore_index=True)

### Finally select nearest link to each SP

In [128]:
nearest_links_remaining = (
    valid_remaining_candidates
    .sort_values("distance_to_link")
    .groupby("No_left")
    .first()
    .reset_index()
)

In [129]:
len(nearest_links_remaining)

105

### Concat 145 stop points (that found a walking link) and 105 stop points that chose the nearest link (outward or fallback)

In [130]:
nearest_links_final = pd.concat([selected_preferred_walk, nearest_links_remaining], ignore_index=True)
nearest_links_final["No_left"].nunique()

250

In [141]:
node_geometry = nodes.set_index("No")["geometry"]

nearest_links_final["from_node_geometry"] = (
    nearest_links_final["FromNode"].map(node_geometry)
)
nearest_links_final["to_node_geometry"] = (
    nearest_links_final["ToNode"].map(node_geometry)
)

# obtener distancia del stop point al FromNode y al ToNode del nearest link seleccionado
nearest_links_final["distance_to_from_node"] = nearest_links_final.apply(
    lambda row: row["stop_geometry"].distance(
        row["from_node_geometry"]
    ),
    axis=1
)

nearest_links_final["distance_to_to_node"] = nearest_links_final.apply(
    lambda row: row["stop_geometry"].distance(
        row["to_node_geometry"]
    ),
    axis=1
)

# si esta mas cerca el FromNode guardar el FromNode
# si esta mas cerca el ToNode guardar el ToNode
nearest_links_final["nearest_node_no"] = np.where(
    nearest_links_final["distance_to_from_node"]
    <= nearest_links_final["distance_to_to_node"],
    
    nearest_links_final["FromNode"],
    nearest_links_final["ToNode"]
)

# obtener la distancia al nodo mas cercano (FromNode o ToNode)
nearest_links_final["distance_to_nearest_node"] = nearest_links_final[
    ["distance_to_from_node", "distance_to_to_node"]
].min(axis=1)

nearest_links_final["nearest_node_geometry"] = (
    nearest_links_final["nearest_node_no"].map(node_geometry)
)

In [142]:
nearest_links_final["distance_to_nearest_node"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
)

count    250.000000
mean      36.672066
std       33.055803
min        4.110088
25%       18.772862
50%       31.603109
75%       44.698420
90%       65.039690
95%       86.253309
99%      164.046191
max      353.089498
Name: distance_to_nearest_node, dtype: float64

In [143]:
nearest_links_final["extra_distance_using_node"] = (
    nearest_links_final["distance_to_nearest_node"]
    - nearest_links_final["distance_to_link"]
)

nearest_links_final["extra_distance_using_node"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
)

count    250.000000
mean      17.103521
std       31.779062
min       -6.160033
25%        1.257690
50%        4.812182
75%       23.888236
90%       44.137891
95%       62.584039
99%      146.259575
max      338.925908
Name: extra_distance_using_node, dtype: float64

In [134]:
nearest_links_final[nearest_links_final["extra_distance_using_node"]>50]

,No_left,NumLines_BRT,NumLines_TL,NodeNo,XCoord,YCoord,geometry,index_right,No_right,FromNode,...,midpoint,outward,selection_method,from_node_geometry,to_node_geometry,distance_to_from_node,distance_to_to_node,nearest_node_no,distance_to_nearest_node,extra_distance_using_node
6,16388.0,3.0,0.0,213463.0,-103.438272,20.635917,"POLYGON ((662814.073 2282636.087, 662813.592 2...",362862,315365.0,129342.0,...,None,NaN,NaN,POINT (662635.878 2282739.185),POINT (662717.277 2282545.517),129.397445,90.626749,129343.0,90.626749,63.608109
74,16466.0,2.0,0.0,213541.0,-103.355941,20.576448,"POLYGON ((671460.32 2276137.398, 671459.839 22...",364114,316633.0,129832.0,...,None,NaN,NaN,POINT (671305.153 2276146.676),POINT (671435.029 2276139.816),55.941881,74.747816,129832.0,55.941881,50.214697
75,16467.0,2.0,0.0,213542.0,-103.355932,20.576550,"POLYGON ((671461.104 2276148.641, 671460.623 2...",364115,316633.0,129833.0,...,None,NaN,NaN,POINT (671435.029 2276139.816),POINT (671305.153 2276146.676),74.449581,55.985507,129832.0,55.985507,50.478382
108,16528.0,0.0,1.0,205646.0,-103.355634,20.707967,"POLYGON ((671344.914 2290697.188, 671344.433 2...",528321,441393.0,182289.0,...,None,NaN,NaN,POINT (671181.553 2290799.199),POINT (671193.423 2290621.058),120.087827,91.907823,182287.0,91.907823,80.611597
109,16529.0,0.0,1.0,205647.0,-103.355601,20.707982,"POLYGON ((671348.398 2290698.862, 671347.916 2...",528320,441393.0,182287.0,...,None,NaN,NaN,POINT (671193.423 2290621.058),POINT (671181.553 2290799.199),95.265750,120.564933,182287.0,95.265750,80.106399
134,16575.0,0.0,1.0,205995.0,-103.355896,20.574988,"POLYGON ((671466.629 2275975.835, 671466.148 2...",364124,316649.0,129838.0,...,None,NaN,NaN,POINT (671441.118 2276096.411),POINT (671331.942 2275931.244),141.729441,56.493623,129839.0,56.493623,55.403507
146,16439.0,2.0,0.0,213514.0,-103.403144,20.739076,"POLYGON ((666362.299 2294091.543, 666361.818 2...",462820,387000.0,158709.0,...,POINT (666265.032 2294088.652),True,outward_nearest,POINT (666400.942 2294226.639),POINT (666097.05 2294137.715),193.579443,171.578157,170220.0,171.578157,82.265418
177,16522.0,0.0,1.0,206352.0,-103.352305,20.730805,"POLYGON ((671665.971 2293228.962, 671665.489 2...",521011,435859.0,179668.0,...,POINT (671567.356 2293228.489),True,outward_nearest,POINT (671652.08 2293538.469),POINT (671495.668 2293066.945),321.262335,176.612323,135791.0,176.612323,158.479379
178,16523.0,0.0,1.0,206353.0,-103.352279,20.730796,"POLYGON ((671668.741 2293228.016, 671668.259 2...",482518,402415.0,165042.0,...,POINT (671567.356 2293228.489),True,outward_nearest,POINT (671519.528 2293081.592),POINT (671672.542 2293548.11),154.472838,336.504643,165042.0,154.472838,143.802552
191,16558.0,0.0,1.0,205676.0,-103.400878,20.607393,"POLYGON ((666741.995 2279516.384, 666741.513 2...",279640,232936.0,92189.0,...,POINT (666643.777 2279515.383),True,outward_nearest,POINT (666839.473 2279809.087),POINT (666350.362 2279021.467),353.089498,574.450069,92189.0,353.089498,338.925908


### Visualize walking links

In [144]:
from shapely.geometry import LineString

walking_links = nearest_links_final.copy()

walking_links["geometry"] = walking_links.apply(
    lambda row: LineString([
        row["stop_geometry"],
        # row["nearest_point"] esto genera el walking link del SP al punto mas cercano del link elegido ("idealmente pero seria partir el link")
        row["nearest_node_geometry"] # esto genera el walking link del SP al nodo mas cercano del link elegido (mas facil para no romper nada en visum)
    ]),
    axis=1
)

walking_links = gpd.GeoDataFrame(
    walking_links,
    geometry="geometry",
    crs=access_stop_points.crs
)

In [145]:
links_map = links.to_crs(4326)
stops_map = access_stop_points.to_crs(4326)
walking_map = walking_links.to_crs(4326)

In [45]:
m = walking_map.explore()
m

In [137]:
nearest_links_map = gpd.GeoDataFrame(
    walking_links,
    geometry="link_geometry",
    crs=links.crs
)
nearest_links_map = nearest_links_map.to_crs(4326)

In [147]:
walking_map.dtypes

No_left                       float64
NumLines_BRT                  float64
NumLines_TL                   float64
NodeNo                        float64
XCoord                        float64
YCoord                        float64
geometry                     geometry
index_right                     int64
No_right                      float64
FromNode                      float64
ToNode                        float64
TSysSet                           str
TypeNo                        float64
NumTouchingLineRoutes         float64
GRUPO_CAP                         str
HIGHWAY                           str
stop_geometry                  object
distance_to_link              float64
outward                        object
selection_method                  str
distance_to_from_node         float64
distance_to_to_node           float64
nearest_node_no               float64
distance_to_nearest_node      float64
extra_distance_using_node     float64
nearest_node_geometry        geometry
dtype: objec

In [ ]:
walking_map = walking_map.drop(columns=["link_geometry", "nearest_point", "midpoint", "from_node_geometry", "to_node_geometry", "nearest_node_geometry"])
walking_map.to_file(os.path.join(PATH, "red_shapefiles", "walking_links_a_nodo.gpkg"))
#stops_map.to_file(os.path.join(PATH, "red_shapefiles", "brt_tl_stops.gpkg"))

In [139]:
nearest_links_map = nearest_links_map.drop(columns=["geometry", "nearest_point", "midpoint", "from_node_geometry", "to_node_geometry"])
nearest_links_map.to_file(os.path.join(PATH, "red_shapefiles", "nearest_links_map.gpkg"))

## __Add walking links to Visum__

In [153]:
nearest_links_final[nearest_links_final["NodeNo"].notna()]

,No_left,NumLines_BRT,NumLines_TL,NodeNo,XCoord,YCoord,geometry,index_right,No_right,FromNode,...,outward,selection_method,from_node_geometry,to_node_geometry,distance_to_from_node,distance_to_to_node,nearest_node_no,distance_to_nearest_node,extra_distance_using_node,nearest_node_geometry
0,16382.0,3.0,0.0,213457.0,-103.433642,20.625220,"POLYGON ((663308.038 2281456.615, 663307.556 2...",359688,312616.0,128199.0,...,NaN,NaN,POINT (663189.893 2281455.458),POINT (663193.249 2281502.021),18.181601,47.754170,128199.0,18.181601,1.533194,POINT (663189.893 2281455.458)
1,16383.0,3.0,0.0,213458.0,-103.433556,20.625260,"POLYGON ((663316.928 2281461.073, 663316.446 2...",310549,260999.0,128201.0,...,NaN,NaN,POINT (663193.249 2281502.021),POINT (663232.922 2281466.854),47.301352,17.007237,104623.0,17.007237,0.396135,POINT (663232.922 2281466.854)
2,16384.0,3.0,0.0,213459.0,-103.441170,20.642951,"POLYGON ((662504.662 2283411.849, 662504.181 2...",358220,311180.0,127550.0,...,NaN,NaN,POINT (662392.643 2283456.627),POINT (662399.309 2283445.044),46.362814,33.623951,127567.0,33.623951,-5.150499,POINT (662399.309 2283445.044)
3,16385.0,3.0,0.0,213460.0,-103.441077,20.642985,"POLYGON ((662514.301 2283415.761, 662513.82 22...",358220,311180.0,127550.0,...,NaN,NaN,POINT (662392.643 2283456.627),POINT (662399.309 2283445.044),46.250688,32.898172,127567.0,32.898172,-4.212589,POINT (662399.309 2283445.044)
4,16386.0,3.0,0.0,213461.0,-103.436846,20.632499,"POLYGON ((662966.356 2282259.175, 662965.875 2...",357295,310240.0,127176.0,...,NaN,NaN,POINT (662888.988 2282161.877),POINT (662894.421 2282214.583),99.895298,52.688918,127175.0,52.688918,29.937239,POINT (662894.421 2282214.583)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,16628.0,2.0,0.0,213590.0,-103.346698,20.622337,"POLYGON ((672372.354 2281227.112, 672371.873 2...",250155,206786.0,81537.0,...,True,outward_nearest,POINT (672237.742 2281310.613),POINT (672275.299 2281203.712),90.390033,23.584238,40358.0,23.584238,16.217701,POINT (672275.299 2281203.712)
246,16629.0,1.0,0.0,213591.0,-103.343440,20.616304,"POLYGON ((672718.637 2280562.702, 672718.156 2...",278368,231509.0,91677.0,...,True,outward_nearest,POINT (672614.962 2280558.715),POINT (672634.506 2280529.043),5.422702,37.212083,91677.0,5.422702,-0.074249,POINT (672614.962 2280558.715)
247,16630.0,1.0,0.0,213592.0,-103.343365,20.616350,"POLYGON ((672726.425 2280567.899, 672725.944 2...",128548,110702.0,44288.0,...,True,outward_nearest,POINT (672674.78 2280512.924),POINT (672569.587 2280662.18),73.214962,110.089095,44288.0,73.214962,66.972349,POINT (672674.78 2280512.924)
248,16633.0,2.0,0.0,213595.0,-103.342941,20.607866,"POLYGON ((672780.224 2279629.144, 672779.743 2...",228506,188751.0,74562.0,...,True,outward_nearest,POINT (672681.854 2279593.137),POINT (672738.773 2279748.462),36.043914,132.908416,74562.0,36.043914,19.117387,POINT (672681.854 2279593.137)


In [158]:
#Red base GDL (con 2,203 zonas)
red_base = os.path.join(PATH, RED_FOLDER, "RedBase 250826 - copia.ver")
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(red_base)
C = com.constants

In [159]:
stations_walking_links = nearest_links_final[["No_left", "NodeNo", "nearest_node_no", "nearest_point", "nearest_node_geometry"]].copy()
stations_walking_links = stations_walking_links.rename(columns={"No_left":"stop_point_id", "NodeNo":"stop_point_node", "nearest_node_no":"nearest_node", "nearest_point":"nearest_point_on_link"})


#Test add walking link
#Visum.Net.AddLink(-1, 213457, 128199, 99)
WALKING_LINK_TYPE = 99

for row in stations_walking_links.itertuples():
    try:
        Visum.Net.AddLink(
            -1,
            row.stop_point_node,
            row.nearest_node,
            WALKING_LINK_TYPE
        )
    except Exception as e:
        print(f"Error creating walking link for stop point {row.stop_point_id}:{e}")

In [156]:
stations_walking_links

,stop_point_id,stop_point_node,nearest_node,nearest_point_on_link,nearest_node_geometry
0,16382.0,213457.0,128199.0,POINT (663191.562 2281454.223),POINT (663189.893 2281455.458)
1,16383.0,213458.0,104623.0,POINT (663231.888 2281468.292),POINT (663232.922 2281466.854)
2,16384.0,213459.0,127567.0,POINT (662401.154 2283450.465),POINT (662399.309 2283445.044)
3,16385.0,213460.0,127567.0,POINT (662401.154 2283450.465),POINT (662399.309 2283445.044)
4,16386.0,213461.0,127175.0,POINT (662860.938 2282237.078),POINT (662894.421 2282214.583)
...,...,...,...,...,...
245,16628.0,213590.0,40358.0,POINT (672265.286 2281225.038),POINT (672275.299 2281203.712)
246,16629.0,213591.0,91677.0,POINT (672614.011 2280559.733),POINT (672614.962 2280558.715)
247,16630.0,213592.0,44288.0,POINT (672631.57 2280571.435),POINT (672674.78 2280512.924)
248,16633.0,213595.0,74562.0,POINT (672696.322 2279623.913),POINT (672681.854 2279593.137)
